<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/Processing/WIKIDATA_DOWNLOAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

תא 1 — התקנות

In [ ]:
!pip -q install "requests>=2.31,<3" "beautifulsoup4>=4.12,<5"

תא 2 — Imports והגדרות

In [ ]:
import collections
import csv
import datetime as dt
import hashlib
import json
import os
import re
import shutil
import time
from pathlib import Path
from urllib.parse import quote

import requests
from bs4 import BeautifulSoup

SEED_QID = "Q184425"
SEED_TITLE = "Pacific War"
YEAR_START = 1941
YEAR_END = 1945

# "us_japan" או "all_pacific"
FOCUS_MODE = "us_japan"

MAX_DEPTH = 2
MAX_QIDS_PER_LAYER = 80
MAX_ARTICLES_PER_LAYER = 80
MAX_CATEGORY_MEMBERS = 80
MAX_CATEGORIES_PER_ARTICLE = 2
MAX_REVERSE_RELATIONS_PER_ITEM = 150
MAX_VALUE_LABELS = 500

LABEL_LANGUAGES = ["en", "ja", "he"]
REQUEST_DELAY_SECONDS = 1.0
REQUEST_TIMEOUT_SECONDS = 45
MAX_RETRIES = 5

# מזהה את המחברת, ללא פרטי קשר אישיים
USER_AGENT = "PacificWarGeoRAGCollector/1.0 (educational research; read-only)"

OUTPUT_DIR = Path("/content/pacific_war_knowledge_v1")
OPTIONAL_SEED_FILE = None
# לדוגמה: OPTIONAL_SEED_FILE = "/content/corpus_seeds.txt"

EN_API = "https://en.wikipedia.org/w/api.php"
WD_API = "https://www.wikidata.org/w/api.php"
WDQS = "https://query.wikidata.org/sparql"

if FOCUS_MODE not in {"us_japan", "all_pacific"}:
    raise ValueError("FOCUS_MODE חייב להיות us_japan או all_pacific")
if MAX_DEPTH < 1 or min(MAX_QIDS_PER_LAYER, MAX_ARTICLES_PER_LAYER) < 1:
    raise ValueError("עומק ומכסים חייבים להיות חיוביים")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_DIR / "_checkpoint_api_cache"
CACHE_DIR.mkdir(exist_ok=True)

EXTRA_QIDS, EXTRA_TITLES = set(), set()
if OPTIONAL_SEED_FILE:
    seed_path = Path(OPTIONAL_SEED_FILE)
    if not seed_path.is_file():
        raise FileNotFoundError(seed_path)
    for line in seed_path.read_text(encoding="utf-8-sig").splitlines():
        value = line.strip()
        if not value or value.startswith("#"):
            continue
        if re.fullmatch(r"Q[1-9]\d*", value):
            EXTRA_QIDS.add(value)
        else:
            EXTRA_TITLES.add(value)

RUN_CONFIG = {
    "schema_version": "1.0",
    "seed_qid": SEED_QID,
    "seed_title": SEED_TITLE,
    "year_start": YEAR_START,
    "year_end": YEAR_END,
    "focus_mode": FOCUS_MODE,
    "max_depth": MAX_DEPTH,
    "max_qids_per_layer": MAX_QIDS_PER_LAYER,
    "max_articles_per_layer": MAX_ARTICLES_PER_LAYER,
    "max_category_members": MAX_CATEGORY_MEMBERS,
    "max_categories_per_article": MAX_CATEGORIES_PER_ARTICLE,
    "max_reverse_relations_per_item": MAX_REVERSE_RELATIONS_PER_ITEM,
    "label_languages": LABEL_LANGUAGES,
    "request_delay_seconds": REQUEST_DELAY_SECONDS,
    "extra_qids": sorted(EXTRA_QIDS),
    "extra_titles": sorted(EXTRA_TITLES),
    "output_dir": str(OUTPUT_DIR),
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()
print(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2))

{
  "schema_version": "1.0",
  "seed_qid": "Q184425",
  "seed_title": "Pacific War",
  "year_start": 1941,
  "year_end": 1945,
  "focus_mode": "us_japan",
  "max_depth": 2,
  "max_qids_per_layer": 80,
  "max_articles_per_layer": 80,
  "max_category_members": 80,
  "max_categories_per_article": 2,
  "max_reverse_relations_per_item": 150,
  "label_languages": [
    "en",
    "ja",
    "he"
  ],
  "request_delay_seconds": 1.0,
  "extra_qids": [],
  "extra_titles": [],
  "output_dir": "/content/pacific_war_knowledge_v1"
}


תא 3 — בקשות, retry ו־checkpoint

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "application/json",
})
LAST_REQUEST_AT = 0.0
ERROR_PATH = OUTPUT_DIR / "errors.jsonl"

def utc_now():
    return dt.datetime.now(dt.timezone.utc).isoformat()

def write_json_atomic(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(
        json.dumps(value, ensure_ascii=False, sort_keys=True),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def log_error(stage, target, error):
    record = {
        "at": utc_now(),
        "stage": stage,
        "target": str(target),
        "error": str(error),
    }
    with ERROR_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    print("ERROR", stage, target, str(error)[:180])

def cached_json(url, params, stage):
    global LAST_REQUEST_AT
    fixed = dict(params)
    if url != WDQS:
        fixed.update({"format": "json", "formatversion": 2, "maxlag": 15})
    cache_key = hashlib.sha256(
        json.dumps([url, fixed], sort_keys=True).encode("utf-8")
    ).hexdigest()
    path = CACHE_DIR / f"{cache_key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    for attempt in range(MAX_RETRIES):
        elapsed = time.monotonic() - LAST_REQUEST_AT
        if elapsed < REQUEST_DELAY_SECONDS:
            time.sleep(REQUEST_DELAY_SECONDS - elapsed)
        try:
            LAST_REQUEST_AT = time.monotonic()
            response = SESSION.get(
                url, params=fixed, timeout=REQUEST_TIMEOUT_SECONDS
            )
            if response.status_code == 429 or response.status_code in {
                500, 502, 503, 504
            }:
                retry_after = response.headers.get("Retry-After")
                try:
                    pause = float(retry_after) if retry_after else 2 ** attempt
                except ValueError:
                    pause = 2 ** attempt
                time.sleep(min(max(pause, 1), 120))
                continue
            response.raise_for_status()
            data = response.json()
            if "error" in data:
                code = data["error"].get("code", "api_error")
                if code == "maxlag":
                    time.sleep(min(2 ** attempt + 2, 120))
                    continue
                raise RuntimeError(json.dumps(data["error"], ensure_ascii=False))
            write_json_atomic(path, data)
            return data
        except (requests.RequestException, ValueError, RuntimeError) as exc:
            if attempt == MAX_RETRIES - 1:
                log_error(stage, fixed, exc)
                raise
            time.sleep(min(2 ** attempt, 60))
    raise RuntimeError(f"Retries exhausted: {stage}")

def batches(values, size):
    values = list(values)
    for start in range(0, len(values), size):
        yield values[start:start + size]

def jsonl_write(path, records):
    tmp = Path(str(path) + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n")
    os.replace(tmp, path)

def jsonl_read(path):
    path = Path(path)
    if not path.exists():
        return []
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

תא 4 — שליפת Wikidata, מאמרים וקשרי גילוי

In [ ]:
ENTITY_CACHE = {}
PAGE_META_CACHE = {}
ARTICLE_CACHE = {}
TRUNCATIONS = []

def fetch_entities(ids):
    missing = sorted(set(ids) - ENTITY_CACHE.keys())
    for group in batches(missing, 20):
        data = cached_json(
            WD_API,
            {
                "action": "wbgetentities",
                "ids": "|".join(group),
                "props": "info|labels|aliases|descriptions|sitelinks|claims",
            },
            "wbgetentities",
        )
        found = data.get("entities", {})
        for item_id in group:
            entity = found.get(item_id)
            if entity and "missing" not in entity:
                ENTITY_CACHE[item_id] = entity
            else:
                log_error("missing_entity", item_id, "Wikidata item missing")
    return {item_id: ENTITY_CACHE.get(item_id) for item_id in ids}

def fetch_page_metadata(titles):
    result = {}
    missing = sorted(set(titles) - PAGE_META_CACHE.keys())

    for group in batches(missing, 20):
        data = cached_json(
            EN_API,
            {
                "action": "query",
                "titles": "|".join(group),
                "redirects": 1,
                "prop": "info|pageprops|revisions",
                "ppprop": "wikibase_item",
                "rvprop": "ids|timestamp",
                "rvslots": "main",
            },
            "page_metadata",
        )

        query = data.get("query", {})
        mapping = {
            row["from"]: row["to"]
            for section in ("normalized", "redirects")
            for row in query.get(section, [])
        }
        pages = {
            page["title"]: page
            for page in query.get("pages", [])
        }

        for title in group:
            canonical = title
            for _ in range(4):
                canonical = mapping.get(canonical, canonical)

            page = pages.get(canonical)
            if page and not page.get("missing") and page.get("ns") == 0:
                PAGE_META_CACHE[title] = page
                PAGE_META_CACHE[canonical] = page
            else:
                log_error(
                    "missing_article",
                    title,
                    "No English main-namespace page",
                )

    for title in titles:
        if title in PAGE_META_CACHE:
            result[title] = PAGE_META_CACHE[title]
    return result

def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for selector in [
        "script", "style", "table", "sup.reference", ".mw-editsection",
        ".navbox", ".reflist", ".metadata", ".hatnote", ".toc",
        ".sidebar", ".thumb", ".mw-empty-elt",
    ]:
        for tag in soup.select(selector):
            tag.decompose()

    root = soup.select_one(".mw-parser-output") or soup
    headings = [
        {"level": tag.name, "text": tag.get_text(" ", strip=True)}
        for tag in root.select("h2, h3, h4")
    ]
    text = root.get_text("\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, headings

def fetch_article(page):
    page_id = page["pageid"]
    if page_id in ARTICLE_CACHE:
        return ARTICLE_CACHE[page_id]

    revisions = page.get("revisions") or []
    if not revisions:
        raise RuntimeError(f"No revision for page {page_id}")

    revision = revisions[0]
    revision_id = revision["revid"]
    data = cached_json(
        EN_API,
        {
            "action": "parse",
            "oldid": revision_id,
            "prop": "wikitext|text|links|categories|tocdata|revid",
        },
        "parse_revision",
    )["parse"]

    if data.get("revid") != revision_id:
        raise RuntimeError(f"Revision mismatch for page {page_id}")

    raw = data["wikitext"]
    html = data["text"]
    if isinstance(raw, dict):
        raw = raw.get("*", "")
    if isinstance(html, dict):
        html = html.get("*", "")

    clean_text, headings = clean_html(html)
    links = sorted({
        link.get("title") or link.get("*")
        for link in data.get("links", [])
        if link.get("ns") == 0
        and (link.get("title") or link.get("*"))
    })
    categories = sorted({
        "Category:" + (cat.get("category") or cat.get("*", ""))
        for cat in data.get("categories", [])
        if cat.get("category") or cat.get("*")
    })

    article = {
        "page_id": page_id,
        "title": page["title"],
        "qid": page.get("pageprops", {}).get("wikibase_item"),
        "revision_id": revision_id,
        "revision_timestamp": revision.get("timestamp"),
        "source_url": (
            f"https://en.wikipedia.org/w/index.php?oldid={revision_id}"
        ),
        "retrieved_at": utc_now(),
        "original_wikitext": raw,
        "original_wikitext_sha256": hashlib.sha256(
            raw.encode("utf-8")
        ).hexdigest(),
        "clean_text": clean_text,
        "clean_text_sha256": hashlib.sha256(
            clean_text.encode("utf-8")
        ).hexdigest(),
        "cleaning_method": (
            "MediaWiki action=parse HTML for the pinned oldid; BeautifulSoup "
            "removes the selectors listed in cell 4, extracts visible text, "
            "and collapses runs of 3+ newlines. Offsets must refer to the "
            "saved clean_text, never to original_wikitext."
        ),
        "headings": headings,
        "links": links,
        "categories": categories,
        "tocdata": data.get("tocdata"),
    }
    ARTICLE_CACHE[page_id] = article
    return article

def category_members(category, cap):
    rows, continuation = [], {}

    while len(rows) < cap:
        data = cached_json(
            EN_API,
            {
                "action": "query",
                "list": "categorymembers",
                "cmtitle": category,
                "cmnamespace": 0,
                "cmtype": "page",
                "cmlimit": 50,
                **continuation,
            },
            "categorymembers",
        )
        rows.extend(
            data.get("query", {}).get("categorymembers", [])
        )
        continuation = data.get("continue", {})
        if not continuation:
            break

    if continuation or len(rows) > cap:
        TRUNCATIONS.append({
            "method": "categorymembers",
            "source": category,
            "cap": cap,
        })
    return rows[:cap]

def reverse_wikidata_parts(qid, cap):
    rows = []
    offset = 0

    while len(rows) < cap:
        query = f"""
        SELECT ?item ?relation WHERE {{
          {{ ?item wdt:P361 wd:{qid} .
             BIND("reverse_part_of" AS ?relation) }}
          UNION
          {{ ?item wdt:P527 wd:{qid} .
             BIND("reverse_has_part" AS ?relation) }}
        }}
        ORDER BY ?item ?relation
        LIMIT 50 OFFSET {offset}
        """
        data = cached_json(
            WDQS,
            {"query": query, "format": "json"},
            "wdqs_reverse_parts",
        )
        batch = data.get("results", {}).get("bindings", [])
        rows.extend(batch)
        if len(batch) < 50:
            break
        offset += 50

    if len(rows) >= cap:
        TRUNCATIONS.append({
            "method": "wdqs_reverse_parts",
            "source": qid,
            "cap": cap,
        })

    result = []
    for row in rows[:cap]:
        target = row["item"]["value"].rsplit("/", 1)[-1]
        if re.fullmatch(r"Q[1-9]\d*", target):
            result.append(
                (target, row["relation"]["value"])
            )
    return result

In [ ]:
def cached_json(url, params, stage):
    global LAST_REQUEST_AT

    fixed = dict(params)
    if url != WDQS:
        fixed.update({"format": "json", "formatversion": 2, "maxlag": 15})

    cache_key = hashlib.sha256(
        json.dumps([url, fixed], sort_keys=True).encode("utf-8")
    ).hexdigest()
    path = CACHE_DIR / f"{cache_key}.json"

    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))

    last_failure = None

    for attempt in range(MAX_RETRIES):
        elapsed = time.monotonic() - LAST_REQUEST_AT
        if elapsed < REQUEST_DELAY_SECONDS:
            time.sleep(REQUEST_DELAY_SECONDS - elapsed)

        try:
            LAST_REQUEST_AT = time.monotonic()
            response = SESSION.get(
                url, params=fixed, timeout=REQUEST_TIMEOUT_SECONDS
            )

            if response.status_code in {429, 500, 502, 503, 504}:
                last_failure = (
                    f"HTTP {response.status_code}; "
                    f"Retry-After={response.headers.get('Retry-After')!r}; "
                    f"response={response.text[:300]!r}"
                )
                retry_after = response.headers.get("Retry-After")
                try:
                    pause = float(retry_after) if retry_after else 2 ** attempt
                except ValueError:
                    pause = 2 ** attempt
                if attempt < MAX_RETRIES - 1:
                    time.sleep(min(max(pause, 1), 120))
                continue

            if not response.ok:
                raise RuntimeError(
                    f"HTTP {response.status_code}; "
                    f"response={response.text[:500]!r}"
                )

            data = response.json()
            if "error" in data:
                error = data["error"]
                code = error.get("code", "api_error")
                last_failure = f"API error {code}: {error.get('info', error)}"
                if code in {"maxlag", "ratelimited"}:
                    if attempt < MAX_RETRIES - 1:
                        time.sleep(min(2 ** attempt + 2, 120))
                    continue
                raise RuntimeError(last_failure)

            write_json_atomic(path, data)
            return data

        except (requests.RequestException, ValueError, RuntimeError) as exc:
            last_failure = str(exc)
            if attempt < MAX_RETRIES - 1:
                time.sleep(min(2 ** attempt, 60))

    message = f"{stage} failed after {MAX_RETRIES} attempts: {last_failure}"
    log_error(stage, fixed, message)
    raise RuntimeError(message)

תא 5 — גילוי, סיווג והכללה

In [ ]:
BLOCKED_QIDS = {"Q362", "Q30", "Q17"}  # WWII, USA, Japan
BLOCKED_TITLES = {"world war ii", "united states", "japan"}

EVENT_RE = re.compile(
    r"\b(battle|campaign|attack|raid|invasion|operation|siege|"
    r"bombing|landing|offensive|war|occupation|surrender)\b", re.I
)
WAR_RE = re.compile(
    r"\b(pacific war|pacific theater|second world war|world war ii|"
    r"world war 2|194[1-5])\b", re.I
)
US_RE = re.compile(
    r"\b(united states|american|u\.s\.|us navy|us army)\b", re.I
)
JP_RE = re.compile(r"\b(japan|japanese|imperial japanese)\b", re.I)
TYPE_PATTERNS = [
    ("ship", re.compile(r"\b(ship|destroyer|aircraft carrier|submarine|battleship)\b", re.I)),
    ("unit_or_organization", re.compile(
        r"\b(army|navy|fleet|division|regiment|squadron|marine corps|air force|command)\b", re.I
    )),
    ("place", re.compile(r"\b(island|atoll|bay|strait|port|city|airfield|peninsula)\b", re.I)),
    ("time", re.compile(r"\b(194[1-5]|period|year)\b", re.I)),
]
RELEVANT_PROPERTIES = {
    "P361": "part_of",
    "P527": "has_part",
    "P710": "participant",
    "P276": "location",
    "P1344": "participant_in",
    "P607": "conflict",
}

def english_value(entity, field):
    return (entity or {}).get(field, {}).get("en", {}).get("value", "")

def claim_qids(entity, property_id):
    values = []
    for claim in (entity or {}).get("claims", {}).get(property_id, []):
        value = claim.get("mainsnak", {}).get("datavalue", {}).get("value")
        if isinstance(value, dict) and re.fullmatch(
            r"Q[1-9]\d*", str(value.get("id", ""))
        ):
            values.append(value["id"])
    return values

def event_years(entity):
    years = []
    # בכוונה אין כאן P569: שנת לידה אינה מסנן.
    for prop in ("P580", "P582", "P585", "P571"):
        for claim in (entity or {}).get("claims", {}).get(prop, []):
            value = claim.get("mainsnak", {}).get("datavalue", {}).get("value", {})
            if isinstance(value, dict):
                match = re.match(r"^[+-](\d{4})", str(value.get("time", "")))
                if match:
                    years.append(int(match.group(1)))
    return years

def item_type(entity, title):
    text = " ".join([
        title,
        english_value(entity, "labels"),
        english_value(entity, "descriptions"),
    ])
    if SEED_QID == (entity or {}).get("id") or EVENT_RE.search(text):
        return "event"
    if "Q5" in claim_qids(entity, "P31"):
        return "person"
    for kind, pattern in TYPE_PATTERNS:
        if pattern.search(text):
            return kind
    return "other"

EDGES = []
EDGE_KEYS = set()
INDEX = {}
SELECTED_QIDS = {SEED_QID}
SELECTED_ARTICLES = {}

def add_edge(source_kind, source_id, relation, depth,
             target_qid=None, target_title=None, category=None):
    record = {
        "source_kind": source_kind,
        "source_id": source_id,
        "relation": relation,
        "depth": depth,
        "target_qid": target_qid,
        "target_title": target_title,
        "category": category,
    }
    key = json.dumps(record, sort_keys=True)
    if key not in EDGE_KEYS:
        EDGE_KEYS.add(key)
        EDGES.append(record)

def classify_candidate(qid, title, entity, article, incoming):
    if qid in BLOCKED_QIDS or title.casefold() in BLOCKED_TITLES:
        return "excluded", "generic_hub_blocked", item_type(entity, title)

    kind = item_type(entity, title)
    if kind == "other":
        return "candidate", "type_not_established", kind

    context = " ".join([
        title,
        english_value(entity, "descriptions"),
        (article or {}).get("clean_text", "")[:4000],
        " ".join((article or {}).get("categories", [])),
    ])
    focused = FOCUS_MODE == "all_pacific" or (
        bool(US_RE.search(context)) and bool(JP_RE.search(context))
    )
    war_signal = bool(WAR_RE.search(context))
    structural = any(edge["relation"] in {
        "part_of", "has_part", "reverse_part_of", "reverse_has_part"
    } for edge in incoming)
    role = any(edge["relation"] in {
        "participant", "location", "participant_in", "conflict"
    } for edge in incoming)
    article_link = any(edge["relation"] in {
        "article_link", "category_member"
    } for edge in incoming)

    if kind == "event":
        years = event_years(entity)
        if years and (max(years) < YEAR_START or min(years) > YEAR_END):
            return "excluded", "event_date_outside_range", kind
        if not (structural or (article_link and war_signal)):
            return "candidate", "no_specific_war_anchor", kind
        if not focused:
            return "candidate", "us_japan_focus_not_established", kind
        if not years and not war_signal and not structural:
            return "candidate", "event_time_not_established", kind
        return "included", (
            "structured_war_link" if structural else "linked_article_with_scope"
        ), kind

    if kind in {"person", "unit_or_organization", "ship", "place", "time"}:
        if role:
            return "included", "structured_role_in_selected_event", kind
        if article_link and war_signal and focused:
            return "included", "linked_article_with_scope", kind
        return "candidate", "insufficient_specific_context", kind

    return "candidate", "unclassified", kind

# זרעי האיסוף: הזרע הראשי נכלל; זרעים אופציונליים עוברים סיווג רגיל.
root_entity = fetch_entities([SEED_QID])[SEED_QID]
if not root_entity:
    raise RuntimeError("Root Wikidata item could not be fetched")
root_page = fetch_page_metadata([SEED_TITLE]).get(SEED_TITLE)
if not root_page:
    raise RuntimeError("Root Wikipedia article could not be fetched")
root_article = fetch_article(root_page)
SELECTED_ARTICLES[root_article["page_id"]] = root_article

INDEX[SEED_QID] = {
    "qid": SEED_QID,
    "label": english_value(root_entity, "labels"),
    "type": "event",
    "status": "included",
    "reason": "explicit_primary_seed",
    "first_depth": 0,
    "enwiki_title": root_article["title"],
    "page_id": root_article["page_id"],
}
for qid in sorted(EXTRA_QIDS):
    add_edge("user_seed", "corpus", "optional_qid_seed", 1, target_qid=qid)
for title in sorted(EXTRA_TITLES):
    add_edge("user_seed", "corpus", "optional_article_seed", 1, target_title=title)

current_qids = {SEED_QID}
current_articles = {root_article["page_id"]: root_article}

for depth in range(MAX_DEPTH):
    print(f"Discovering layer {depth + 1}/{MAX_DEPTH}")
    layer_start = len(EDGES)

    for qid in sorted(current_qids):
        entity = fetch_entities([qid]).get(qid)
        if not entity:
            continue
        for prop, relation in RELEVANT_PROPERTIES.items():
            for target in sorted(set(claim_qids(entity, prop))):
                add_edge("wikidata_item", qid, relation, depth + 1,
                         target_qid=target)
        if INDEX.get(qid, {}).get("type") == "event":
            try:
                for target, relation in reverse_wikidata_parts(
                    qid, MAX_REVERSE_RELATIONS_PER_ITEM
                ):
                    add_edge("wikidata_item", qid, relation, depth + 1,
                             target_qid=target)
            except Exception as exc:
                log_error("reverse_wikidata_parts", qid, exc)

    for article in sorted(current_articles.values(), key=lambda x: x["page_id"]):
        for title in article["links"]:
            add_edge("wikipedia_article", article["page_id"],
                     "article_link", depth + 1, target_title=title)
        matching_categories = [
            category for category in article["categories"]
            if re.search(r"Pacific War|World War II.*Pacific|Battles.*Pacific", category, re.I)
        ][:MAX_CATEGORIES_PER_ARTICLE]
        for category in matching_categories:
            try:
                for member in category_members(category, MAX_CATEGORY_MEMBERS):
                    add_edge(
                        "wikipedia_article", article["page_id"],
                        "category_member", depth + 1,
                        target_title=member["title"], category=category
                    )
            except Exception as exc:
                log_error("category_members", category, exc)

    layer_edges = EDGES[layer_start:]
    if depth == 0:
        layer_edges += [
            edge for edge in EDGES[:layer_start]
            if edge["source_kind"] == "user_seed"
        ]

    all_titles = sorted({
        edge["target_title"] for edge in layer_edges if edge["target_title"]
    })
    allowed_titles = all_titles[:MAX_ARTICLES_PER_LAYER]
    if len(all_titles) > len(allowed_titles):
        TRUNCATIONS.append({
            "method": "article_candidates",
            "source": f"layer_{depth + 1}",
            "cap": MAX_ARTICLES_PER_LAYER,
            "unfetched": len(all_titles) - len(allowed_titles),
        })
    metadata = fetch_page_metadata(allowed_titles)
    title_to_page = {}
    for title, page in metadata.items():
        title_to_page[title] = page
        title_to_page[page["title"]] = page

    for edge in layer_edges:
        page = title_to_page.get(edge["target_title"])
        if page:
            edge["target_title"] = page["title"]
            edge["target_page_id"] = page["pageid"]
            edge["target_qid"] = edge["target_qid"] or (
                page.get("pageprops", {}).get("wikibase_item")
            )

    qids = sorted({
        edge["target_qid"] for edge in layer_edges if edge["target_qid"]
    })
    allowed_qids = qids[:MAX_QIDS_PER_LAYER]
    if len(qids) > len(allowed_qids):
        TRUNCATIONS.append({
            "method": "qid_candidates",
            "source": f"layer_{depth + 1}",
            "cap": MAX_QIDS_PER_LAYER,
            "unfetched": len(qids) - len(allowed_qids),
        })
    try:
        fetch_entities(allowed_qids)
    except Exception as exc:
        log_error("entity_batch_layer", depth + 1, exc)

    candidate_keys = set(allowed_qids)
    for title in allowed_titles:
        page = title_to_page.get(title)
        if page and not page.get("pageprops", {}).get("wikibase_item"):
            candidate_keys.add("TITLE:" + page["title"])

    next_qids = set()
    next_articles = {}
    for key in sorted(candidate_keys):
        qid = None if key.startswith("TITLE:") else key
        entity = ENTITY_CACHE.get(qid) if qid else None
        if qid and entity is None:
            INDEX[qid] = {
                "qid": qid, "label": None, "type": "unknown",
                "status": "fetch_failed", "reason": "entity_unavailable",
                "first_depth": depth + 1, "enwiki_title": None, "page_id": None,
            }
            continue

        matching_pages = [
            page for page in title_to_page.values()
            if (qid and page.get("pageprops", {}).get("wikibase_item") == qid)
            or (not qid and "TITLE:" + page["title"] == key)
        ]
        if qid and not matching_pages:
            sitelink = entity.get("sitelinks", {}).get("enwiki", {})
            linked_title = sitelink.get("title")
            if linked_title:
                matching_pages = list(fetch_page_metadata([linked_title]).values())
        page = matching_pages[0] if matching_pages else None
        title = page["title"] if page else (
            entity.get("sitelinks", {}).get("enwiki", {}).get("title")
            or english_value(entity, "labels") or key
        )
        article = None
        if page:
            try:
                article = fetch_article(page)
            except Exception as exc:
                log_error("fetch_article", title, exc)

        incoming = [
            edge for edge in layer_edges
            if (qid and edge.get("target_qid") == qid)
            or (not qid and edge.get("target_title") == title)
        ]
        status, reason, kind = classify_candidate(
            qid, title, entity, article, incoming
        )
        if qid:
            existing = INDEX.get(qid)
            if existing and existing["status"] == "included":
                continue
            INDEX[qid] = {
                "qid": qid,
                "label": english_value(entity, "labels"),
                "type": kind,
                "status": status,
                "reason": reason,
                "first_depth": depth + 1,
                "enwiki_title": title if page else None,
                "page_id": page["pageid"] if page else None,
            }
        if status == "included":
            if qid:
                SELECTED_QIDS.add(qid)
                if kind == "event":
                    next_qids.add(qid)
            if article:
                SELECTED_ARTICLES[article["page_id"]] = article
                if kind == "event":
                    next_articles[article["page_id"]] = article

    # מועמדים שהמכסה מנעה שליפה נשארים גלויים באינדקס.
    for qid in qids[MAX_QIDS_PER_LAYER:]:
        INDEX.setdefault(qid, {
            "qid": qid, "label": None, "type": "unknown",
            "status": "candidate", "reason": "qid_layer_budget",
            "first_depth": depth + 1, "enwiki_title": None, "page_id": None,
        })

    current_qids = next_qids
    current_articles = next_articles
    print(
        "Selected QIDs:", len(SELECTED_QIDS),
        "selected articles:", len(SELECTED_ARTICLES),
        "indexed QIDs:", len(INDEX),
    )

Discovering layer 1/2
ERROR missing_article BUROGU No English main-namespace page
Selected QIDs: 28 selected articles: 28 indexed QIDs: 239
Discovering layer 2/2
Selected QIDs: 40 selected articles: 40 indexed QIDs: 456


תא 6 — יצוא הפריטים, ה־statements, המאמרים והדוחות

In [ ]:
selected_entities = fetch_entities(sorted(SELECTED_QIDS))
selected_entities = {
    qid: entity for qid, entity in selected_entities.items() if entity
}

# Labels של ערכי QID ושל מאפייני P, עד מכסה מפורשת.
value_qids = set()
property_ids = set()
for entity in selected_entities.values():
    for prop, claims in entity.get("claims", {}).items():
        property_ids.add(prop)
        for claim in claims:
            value = claim.get("mainsnak", {}).get("datavalue", {}).get("value")
            if isinstance(value, dict) and re.fullmatch(
                r"Q[1-9]\d*", str(value.get("id", ""))
            ):
                value_qids.add(value["id"])

missing_value_labels = sorted(value_qids - ENTITY_CACHE.keys())
label_targets = missing_value_labels[:MAX_VALUE_LABELS]
if len(missing_value_labels) > len(label_targets):
    TRUNCATIONS.append({
        "method": "value_qid_labels",
        "source": "selected_statements",
        "cap": MAX_VALUE_LABELS,
        "unfetched": len(missing_value_labels) - len(label_targets),
    })
for group in batches(label_targets + sorted(property_ids), 20):
    try:
        fetch_entities(group)
    except Exception as exc:
        log_error("value_label_batch", "|".join(group), exc)

retrieved_at = utc_now()
item_rows = [
    {
        "qid": qid,
        "retrieved_at": retrieved_at,
        "lastrevid": entity.get("lastrevid"),
        "entity": entity,  # תגובת הפריט המלאה של wbgetentities
    }
    for qid, entity in sorted(selected_entities.items())
]
jsonl_write(OUTPUT_DIR / "wikidata_items.jsonl", item_rows)

statement_rows = []
for qid, entity in sorted(selected_entities.items()):
    for prop, claims in sorted(entity.get("claims", {}).items()):
        for position, claim in enumerate(claims):
            mainsnak = claim.get("mainsnak", {})
            value = mainsnak.get("datavalue", {}).get("value")
            value_qid = value.get("id") if isinstance(value, dict) else None
            statement_rows.append({
                "qid": qid,
                "property_id": prop,
                "property_label_en": english_value(
                    ENTITY_CACHE.get(prop), "labels"
                ) or None,
                "statement_id": claim.get("id"),
                "statement_position": position,
                "rank": claim.get("rank"),
                "snaktype": mainsnak.get("snaktype"),
                "datatype": mainsnak.get("datatype"),
                "value": value,
                "value_qid": value_qid,
                "value_label_en": english_value(
                    ENTITY_CACHE.get(value_qid), "labels"
                ) if value_qid else None,
                "qualifier_property_ids": sorted(claim.get("qualifiers", {})),
                "reference_count": len(claim.get("references", [])),
                "relevant_for_discovery": prop in RELEVANT_PROPERTIES,
                "raw_item_qid": qid,
            })
jsonl_write(OUTPUT_DIR / "statements.jsonl", statement_rows)

article_rows = [
    SELECTED_ARTICLES[page_id]
    for page_id in sorted(SELECTED_ARTICLES)
]
jsonl_write(OUTPUT_DIR / "wikipedia_articles.jsonl", article_rows)

edge_rows = sorted(
    EDGES,
    key=lambda x: (
        x["depth"], x["source_kind"], str(x["source_id"]),
        x["relation"], str(x.get("target_qid")),
        str(x.get("target_title")),
    ),
)
jsonl_write(OUTPUT_DIR / "discovery_edges.jsonl", edge_rows)

index_rows = [INDEX[qid] for qid in sorted(INDEX)]
jsonl_write(OUTPUT_DIR / "item_index.jsonl", index_rows)

status_counts = dict(sorted(collections.Counter(
    row["status"] for row in index_rows
).items()))
method_targets = collections.defaultdict(set)
for edge in edge_rows:
    target = edge.get("target_qid") or edge.get("target_title")
    if target:
        method_targets[edge["relation"]].add(str(target))
discovered_by_method = {
    method: len(targets)
    for method, targets in sorted(method_targets.items())
}
error_rows = jsonl_read(ERROR_PATH)

quality = {
    "status_counts": status_counts,
    "discovered_by_method_unique_targets": discovered_by_method,
    "wikidata_items_saved": len(item_rows),
    "wikipedia_articles_saved": len(article_rows),
    "statements_saved": len(statement_rows),
    "discovery_edges_saved": len(edge_rows),
    "errors_logged": len(error_rows),
    "truncations": TRUNCATIONS,
    "items_without_enwiki_article": sum(
        1 for row in index_rows
        if row["status"] == "included" and not row["page_id"]
    ),
    "items_with_no_event_date_property": sum(
        1 for qid, entity in selected_entities.items()
        if INDEX[qid]["type"] == "event" and not event_years(entity)
    ),
}
write_json_atomic(OUTPUT_DIR / "quality_summary.json", quality)

manifest = {
    "schema_version": "1.0",
    "config_hash": CONFIG_HASH,
    "configuration": RUN_CONFIG,
    "generated_at": utc_now(),
    "api_sources": [EN_API, WD_API, WDQS],
    "wikidata_revisions": {
        qid: entity.get("lastrevid")
        for qid, entity in sorted(selected_entities.items())
    },
    "wikipedia_revisions": {
        str(article["page_id"]): article["revision_id"]
        for article in article_rows
    },
    "counts": {
        "indexed_qids": len(index_rows),
        "selected_qids": len(item_rows),
        "selected_articles": len(article_rows),
        "statements": len(statement_rows),
        "discovery_edges": len(edge_rows),
    },
    "coverage_boundary": (
        "Bounded traversal from Q184425 and the English Pacific War article, "
        "plus optional corpus seeds; limited by depth and per-layer caps. "
        "Only selected event nodes are expanded. Generic hubs are blocked. "
        "Wikipedia links and category membership are discovery signals, "
        "not evidence of a factual relation or a claim in project text. "
        "WDQS reverse relations and category membership may reflect a later "
        "snapshot than the pinned article revision. This is not an exhaustive "
        "list of Pacific War items."
    ),
    "not_covered": [
        "items beyond configured depth or caps",
        "items absent from chosen links, categories, and Wikidata relations",
        "items missed because source fields or article QIDs are absent",
        "failed API requests",
        "paragraph evidence, NER spans, silver labels, and entity linking decisions",
    ],
    "files": [
        "wikidata_items.jsonl", "statements.jsonl",
        "wikipedia_articles.jsonl", "discovery_edges.jsonl",
        "item_index.jsonl", "quality_summary.json",
        "errors.jsonl",
    ],
}
write_json_atomic(OUTPUT_DIR / "manifest.json", manifest)
print(json.dumps(quality, ensure_ascii=False, indent=2))